# Read and Merge All Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, root)
from server.utils.database import engine


KeyboardInterrupt



In [ ]:
 # Load rental listings
load_sql = """
SELECT rl.listing_db_id, rl.latitude, rl.longitude, rl.price, aq.aqi, gn.nwi_score
FROM rental_listings rl
JOIN listing_clusters lc on rl.listing_db_id = lc.listing_db_id
JOIN cluster_air_quality aq on lc.cluster_id = aq.cluster_id
JOIN listings_geo lg on rl.listing_db_id = lg.listing_db_id
JOIN geo_nwi gn on lg.geo_id = gn.geo_id

WHERE
  rl.state = 'DC'
  OR
  (rl.state = 'MD' AND rl.county IN ('Montgomery', 'Prince George''s'))
  OR
  (rl.state = 'VA' AND (rl.county IN ('Arlington', 'Fairfax', 'Loudoun')
                     OR rl.city IN ('Alexandria', 'Fairfax', 'Falls Church')
                    )
  )
ORDER BY lc.listing_db_id;

"""
rental_units = pd.read_sql(load_sql, engine)
print(rental_units.shape)

In [ ]:
nearest_bus_read_sql = """
SELECT
  rl.listing_db_id,
    ROUND((ST_Distance(rl.geom::geography, bs.geom::geography) / 1609.34)::numeric, 2) AS nearest_bus_stop_miles
FROM
  public.rental_listings rl
JOIN LATERAL (
  SELECT bs.id, bs.name, bs.geom
  FROM public.bus_stops bs
  ORDER BY rl.geom <-> bs.geom  -- KNN search: uses spatial index!
  LIMIT 1
) bs ON TRUE
WHERE
  state = 'DC'
  OR
  (state = 'MD' AND county IN ('Montgomery', 'Prince George''s'))
  OR
  (state = 'VA' AND (county IN ('Arlington', 'Fairfax', 'Loudoun')
                     -- Assuming independent cities are stored in the 'county' or a similar field, adjust as needed:
                     OR city IN ('Alexandria', 'Fairfax', 'Falls Church')
                    )
  );
"""
nearest_bus = pd.read_sql(nearest_bus_read_sql, engine)
print(nearest_bus.shape)

In [ ]:
count_bus_stops_read_sql = """
SELECT
  rl.listing_db_id,
  COUNT(bs.id) AS nearby_bus_stops
FROM
  public.rental_listings rl
LEFT JOIN
  public.bus_stops bs
  ON ST_DWithin(rl.geom, bs.geom, 0.0145)
WHERE
  state = 'DC'
  OR
  (state = 'MD' AND county IN ('Montgomery', 'Prince George''s'))
  OR
  (state = 'VA' AND (county IN ('Arlington', 'Fairfax', 'Loudoun')
                     -- Assuming independent cities are stored in the 'county' or a similar field, adjust as needed:
                     OR city IN ('Alexandria', 'Fairfax', 'Falls Church')
                    )
  )
GROUP BY
  rl.listing_db_id,
  rl.listing_name
ORDER BY
  nearby_bus_stops DESC;
"""

count_bus_stops = pd.read_sql(count_bus_stops_read_sql, engine)
print(count_bus_stops.shape)

In [ ]:
count_park_read_sql = """
SELECT
  rl.listing_db_id,
  COUNT(os.id) AS nearby_parks
FROM
  rental_listings rl
LEFT JOIN
  open_street os
  ON ST_DWithin(rl.geom, os.geom, 0.0145)
WHERE
  state = 'DC'
  OR
  (state = 'MD' AND county IN ('Montgomery', 'Prince George''s'))
  OR
  (state = 'VA' AND (county IN ('Arlington', 'Fairfax', 'Loudoun')
                     -- Assuming independent cities are stored in the 'county' or a similar field, adjust as needed:
                     OR city IN ('Alexandria', 'Fairfax', 'Falls Church')
                    )
  )
GROUP BY
  rl.listing_db_id
ORDER BY
  nearby_parks DESC;
"""

count_park = pd.read_sql(count_park_read_sql, engine)
print(count_park.shape)

In [ ]:
closest_park_read_sql = """
SELECT
  rl.listing_db_id,
  ROUND((ST_Distance(rl.geom::geography, os.geom::geography) / 1609.34)::numeric, 2) AS nearest_park_miles
FROM
  rental_listings rl
JOIN LATERAL (
  SELECT os.id, os.name, os.geom
  FROM open_street os
  WHERE os.leisure = 'park'
  ORDER BY rl.geom <-> os.geom  -- fast KNN using index
  LIMIT 1
) os ON TRUE
WHERE
  state = 'DC'
  OR
  (state = 'MD' AND county IN ('Montgomery', 'Prince George''s'))
  OR
  (state = 'VA' AND (county IN ('Arlington', 'Fairfax', 'Loudoun')
                     -- Assuming independent cities are stored in the 'county' or a similar field, adjust as needed:
                     OR city IN ('Alexandria', 'Fairfax', 'Falls Church')
                    )
  );
"""

closest_park = pd.read_sql(closest_park_read_sql, engine)
print(closest_park.shape)

In [ ]:
# Merge all dataframes

merged_df = rental_units.merge(nearest_bus, on="listing_db_id", how="left")
merged_df = merged_df.merge(count_bus_stops, on="listing_db_id", how="left")
merged_df = merged_df.merge(count_park, on="listing_db_id", how="left")
merged_df = merged_df.merge(closest_park, on="listing_db_id", how="left")

In [ ]:
merged_df.head(3)

In [ ]:
merged_df.to_csv("temp.csv", index=False)

In [ ]:
working_df = merged_df.copy()

# Exploratory Data Analysis (EDA)

### Data Overview

In [ ]:
print("Dataset Information:")
print(working_df.info())
print("\nSummary Statistics:")
print(working_df.describe())
print("\nMissing Values:")
print(working_df.isnull().sum())

### Distribution of Features

In [ ]:
# Setting up the plot style
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1)
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]

plotting_cols = ["price", "aqi", "nwi_score", "nearest_bus_stop_miles", "nearby_bus_stops", "nearby_parks", "nearest_park_miles"]
plt.figure(figsize=(10, 8))
for i, col in enumerate(plotting_cols, 1):
    plt.subplot(3, 3, i)
    sns.histplot(working_df[col], bins=30, kde=True)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

plt.tight_layout()
plt.savefig("f1.png", dpi=300, bbox_inches="tight")

In [ ]:
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1)
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]
plt.figure(figsize=(10, 8))
for i, col in enumerate(plotting_cols, 1):
    plt.subplot(3, 3, i)
    sns.boxplot(x=col, data=working_df)
    plt.title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()

plt.tight_layout()
plt.savefig("f2.png", dpi=300, bbox_inches="tight")

1. Price:
* Observation: The distribution is heavily right-skewed, with a peak around 2000–3000 and a long tail extending past 15,000. Most rental prices are on the lower end, but there are some extreme values (outliers).
* Insight: This skewness is expected for features like rental price, as most units are affordable, but a few luxury units can skew the distribution.
* Action: Should do log transformation
2. AQI:
* Observation: The Air Quality Index (AQI) distribution is bimodal, with peaks around 35 and 45. There are also smaller peaks at lower values (around 10–20).
* Insight: There are two distinct groups of air quality levels, possibly reflecting urban vs. suburban clusters (as the proposal mentions AQI is aggregated at the cluster level via HDBSCAN). AQI values are generally moderate (EPA categories: 0–50 is "Good," 51–100 is "Moderate"), which is reasonable for the Washington D.C. Metro Area.
* Action: No immediate action needed
3. NWI Score:
* Observation: The NWI score distribution is right-skewed, with a peak around 15 (Above Average to Most Walkable) and a smaller peak around 5-10 (Moderate to Above Average).
* Insight: This suggests that most rental listings are in areas with moderate NWI scores, while a few listings are in areas with very high NWI or very low NWI scores. The walkability setting match urban areas' characteristics.
* Action: Since NWI scores are already standardized, no action is needed.
4. Nearest Bus Stop Distance:
* Observation: The distance to the nearest bus stop (in miles) is heavily right-skewed, with a peak near 0 and a long tail extending to 300 miles. Most distances are small, but there are some extreme values.
* Insight: Most housing units are close to a bus stop, which is expected in a metro area like Washington D.C. The extreme values (e.g., 300 miles) are likely outliers or errors, as it’s unrealistic for a metro area housing unit to be that far from a bus stop.
* Action: Should do log transformation
5. Nearby Bus Stops:
* Observation: The count of bus stops within a 1-mile radius is also right-skewed, with a peak near 0–50 and a tail extending to 250. Most units have fewer nearby bus stops, but some have many.
* Insight: This distribution reflects urban density—some areas (likely downtown) have high bus stop density, while others (suburban/rural) have fewer. The skewness is expected, and standardization will be applied.
* Action: Consider log transformation
6. Nearby Parks:
* Observation: The count of parks within a 1-mile radius is discrete and right-skewed, with peaks at 0, 1, and 2 parks, and a tail extending to 6. Most units have 0–2 nearby parks.
* Insight: This suggests limited access to green spaces for most units, with a few having more parks nearby. The discrete nature (integers) is expected for a count variable.
* Action: No immediate action needed
7. Nearest Park Distance:
* Observation: The distance to the nearest park (in miles) is right-skewed, with a peak near 0 and a tail extending to 300 miles. Most distances are small, but there are extreme values.
* Insight: Similar to nearest_bus_stop_miles, most units are close to a park, but the extreme values (e.g., 300 miles) are suspicious.
* Action: Should do log transformation

### Correlation Matrix

In [ ]:
corr_matrix = working_df[plotting_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True, vmin=-1,  vmax=1)
plt.title("Correlation Matrix")
plt.show()
plt.tight_layout()
plt.savefig("f3.png", dpi=300, bbox_inches="tight")


### Pairwise Scatter Plots

In [ ]:
sns.pairplot(working_df[plotting_cols])
plt.suptitle("Pairwise Scatter Plots", y=1.02)
plt.tight_layout()
plt.show()
plt.savefig("f4.png", dpi=300, bbox_inches="tight")

#### Investigate the Correlation
Let's try without rounding

In [ ]:
sql = """
SELECT
  rl.listing_db_id,
  (ST_Distance(rl.geom::geography, bs.geom::geography) / 1609.34) AS nearest_bus_stop_miles,
  (ST_Distance(rl.geom::geography, os.geom::geography) / 1609.34) AS nearest_park_miles
FROM
  rental_listings rl
JOIN LATERAL (
  SELECT bs.id, bs.name, bs.geom
  FROM public.bus_stops bs
  ORDER BY rl.geom <-> bs.geom
  LIMIT 1
) bs ON TRUE
JOIN LATERAL (
  SELECT os.id, os.name, os.geom
  FROM open_street os
  WHERE os.leisure = 'park'
  ORDER BY rl.geom <-> os.geom
  LIMIT 1
) os ON TRUE;
"""
df = pd.read_sql(sql, engine)
correlation = df["nearest_bus_stop_miles"].corr(df["nearest_park_miles"])
print(f"Correlation between nearest bus stop and nearest park distance without rounding: {correlation}")

Let's try with 5 miles instead of 1 miles

In [ ]:
sql_5_miles = """
SELECT
  rl.listing_db_id,
  LEAST((ST_Distance(rl.geom::geography, bs.geom::geography) / 1609.34), 5.0) AS nearest_bus_stop_miles,
  LEAST((ST_Distance(rl.geom::geography, os.geom::geography) / 1609.34), 5.0) AS nearest_park_miles
FROM
  rental_listings rl
JOIN LATERAL (
  SELECT bs.id, bs.name, bs.geom
  FROM public.bus_stops bs
  ORDER BY rl.geom <-> bs.geom
  LIMIT 1
) bs ON TRUE
JOIN LATERAL (
  SELECT os.id, os.name, os.geom
  FROM open_street os
  WHERE os.leisure = 'park'
  ORDER BY rl.geom <-> os.geom
  LIMIT 1
) os ON TRUE;
"""
df = pd.read_sql(sql_5_miles, engine)
correlation = df["nearest_bus_stop_miles"].corr(df["nearest_park_miles"])
print(f"Correlation between nearest bus stop and nearest park distance without rounding: {correlation}")

### Check Geospatial Heatmap

In [ ]:
import folium
from folium.plugins import HeatMap

df_bus = working_df[["latitude", "longitude", "nearest_bus_stop_miles"]].copy()

center_lat_bus = df_bus["latitude"].mean()
center_lon_bus = df_bus["longitude"].mean()

m_nearest_bus = folium.Map(location=[center_lat_bus, center_lon_bus], zoom_start=8)
heat_data_bus = df_bus[["latitude", "longitude", "nearest_bus_stop_miles"]].values.tolist()

HeatMap(heat_data_bus, radius=15).add_to(m_nearest_bus)

title_html_bus = """
<h3 align="center" style="font-size:20px"><b>Heatmap of Nearest Bus Stop Distance</b></h3>
"""
map1_filename = "nearest_bus_stop_heatmap.html"
m_nearest_bus.save(map1_filename)
# Display the map
m_nearest_bus

In [ ]:
import folium
from folium.plugins import HeatMap
import math

df_park = working_df[["latitude", "longitude", "nearest_park_miles"]].copy()

center_lat_park = df_park["latitude"].mean()
center_lon_park = df_park["longitude"].mean()

m_nearest_park = folium.Map(location=[center_lat_park, center_lon_park], zoom_start=8)
heat_data_park = df_park[["latitude", "longitude", "nearest_park_miles"]].values.tolist()

HeatMap(heat_data_park, radius=15).add_to(m_nearest_park)

title_html_park = """
<h3 align="center" style="font-size:20px"><b>Heatmap of Nearest Park Distance</b></h3>
"""
map2_filename = "nearest_park_heatmap.html"
m_nearest_park.save(map2_filename)
# Display the map
m_nearest_park

### Conclusion
* From the geospatial heatmaps, we can see that the nearest bus stop and nearest park distances are identical for many listings, which is likely due to the data sparsity and urban planning patterns in the Washington D.C. Metro Area.

### Analysis of High Correlation Between Distance Features
---

**Observation:** A very high correlation (r = 0.95) was observed between `nearest_bus_stop_miles` and `nearest_park_miles`, even after capping distances at 5 miles to mitigate outlier effects.

**Potential Reasons for High Correlation:**

1.  **Spatial Co-location:** Urban planning often places parks and bus routes in proximity. Parks might be situated near transit corridors, or bus stops may be located at park entrances or along bordering roads. This spatial relationship naturally leads to similar distances from a given listing to the nearest park and the nearest bus stop.
2.  **Limited Number of Unique Parks:** The underlying park data might contain a limited number of distinct park locations relative to the number of listings. Consequently, many listings share the same nearest park, leading to identical `nearest_park_miles` values for those listings. If these listings also share similar `nearest_bus_stop_miles` values, it reinforces the correlation.

---

### Rationale for Retaining Both Features for PCA

While `nearest_bus_stop_miles` and `nearest_park_miles` are highly correlated (r = 0.95), combining or dropping one of them is not necessarily required or optimal when using Principal Component Analysis (PCA). Here’s the rationale for retaining both:

1.  **PCA Handles Multicollinearity:** PCA is specifically designed to handle multicollinearity. Highly correlated variables represent shared variance, which PCA effectively captures by creating principal components where these variables have significant loadings. The algorithm naturally identifies and represents this underlying dimension (e.g., "proximity to amenities" or "urban core distance") without needing manual feature combination.
2.  **Avoiding Information Loss and Assumptions:** Combining features (e.g., by averaging) imposes an assumption about their relative importance and relationship. Keeping both allows the PCA algorithm to determine the optimal weighting based purely on the data's variance structure, potentially capturing nuances that a simple combination might obscure. While they share ~90% of their variance (\(R^2 \approx 0.90\)), retaining both preserves the unique 10% variance each contributes.
3.  **Standard Practice:** In many applications, it's standard practice to retain correlated features when performing PCA, as the technique itself serves as a dimensionality reduction method that addresses the redundancy. Dropping features is more critical for methods sensitive to multicollinearity in coefficient estimation (like standard linear regression) or when features are perfectly collinear or theoretically identical.
4.  **Interpretability:** While combining *might* seem simpler, interpreting PCA components often involves understanding which original variables load heavily onto them. Seeing both distance measures load onto a component clearly signals that it represents spatial proximity, which is directly interpretable.

---

### Decision: Retain Both Correlated Features

Based on PCA's ability to handle multicollinearity and the desire to let the data structure guide the dimensionality reduction, we decided to **retain both** `nearest_bus_stop_miles` and `nearest_park_miles` as separate input features for the PCA.

- **Method:** Both features will be included in the dataset used for PCA after appropriate transformations (e.g., log), directionality adjustments (inverting distances so higher means better), and scaling.
- **Rationale:** This approach avoids imposing assumptions through manual combination and allows PCA to determine the contribution of each distance measure based on the observed variance.

**Final QoL Feature Set (Proposed for PCA Input):**

* `price_log` (Log-transformed price)
* `aqi` (Air Quality Index - potentially inverted)
* `nwi_score` (Walkability Score)
* `nearby_bus_stops` (Count of nearby stops)
* `nearby_parks` (Count of nearby parks)
* `nearest_bus_stop_miles` (Distance to nearest bus stop)
* `nearest_park_miles` (Distance to nearest park)

In [ ]:
# --- Plot 1: nearby_parks ---
df_park = working_df[["latitude", "longitude", "nearby_parks"]].copy()

# Calculate center of the map
center_lat_parks = df_park["latitude"].mean()
center_lon_parks = df_park["longitude"].mean()

# Create map object
m_parks = folium.Map(location=[center_lat_parks, center_lon_parks], zoom_start=8)

# Create heat map data
heat_data_parks = df_park[["latitude", "longitude", "nearby_parks"]].values.tolist()

# Add heat map layer
HeatMap(heat_data_parks, radius=10, blur=5).add_to(m_parks)

# Add title
title_html_parks = """
             <h3 align="center" style="font-size:16px"><b>Heatmap of Nearby Parks Count</b></h3>
             """
m_parks.get_root().html.add_child(folium.Element(title_html_parks))

# Save map (optional)
map_parks_filename = "heatmap_nearby_parks.html"
m_parks.save(map_parks_filename)

# Display map in Jupyter
m_parks

In [ ]:
# --- Plot 2: nearby_bus_stops ---
df_bus = working_df[["latitude", "longitude", "nearby_bus_stops"]].copy()
# Calculate center of the map
center_lat_bus = df_bus["latitude"].mean()
center_lon_bus = df_bus["longitude"].mean()

# Create map object
m_bus = folium.Map(location=[center_lat_bus, center_lon_bus], zoom_start=8)

# Create heat map data
heat_data_bus = df_bus[["latitude", "longitude", "nearby_bus_stops"]].values.tolist()

# Add heat map layer
HeatMap(heat_data_bus, radius=10, blur=5).add_to(m_bus)

# Add title
title_html_bus = """
             <h3 align="center" style="font-size:16px"><b>Heatmap of Nearby Bus Stops Count</b></h3>
             """
m_bus.get_root().html.add_child(folium.Element(title_html_bus))

# Save map (optional)
map_bus_filename = "heatmap_nearby_bus_stops.html"
m_bus.save(map_bus_filename)

# Display map in Jupyter
m_bus

### Investigate nearby_bus_stops and nearby_parks
We think that the correlation of 0.77 between `nearby_bus_stops` and `nearby_parks` reflects a real-world relationship of urban density that we do not want to obscure.
Therefore, we decide to keep both features, and only apply log transformation to `nearby_bus_stops` to reduce the skewness.

### Plot Other Maps Just For Fun

#### Walkabiliy, Air Quality and Rental Price Around VT Innovation Campus

In [ ]:
import branca.colormap as cm
import math

# configuration
vt_ic_lat = 38.8334
vt_ic_lon = -77.0464
zoom_level = 10

color_features = ["nwi_score", "aqi", "price"]

required_cols = ["listing_db_id", "latitude", "longitude"] + color_features
plot_df = working_df[required_cols].copy()

all_maps = {}

for feature in color_features:
    # Create a new folium map centered around the VT Innovation Campus
    m = folium.Map(location=[vt_ic_lat, vt_ic_lon], zoom_start=zoom_level)

    # Create a colormap for the feature
    min_val = plot_df[feature].min()
    max_val = plot_df[feature].quantile(0.95)
    mid_val = plot_df[feature].median()
    
    if feature == "nwi_score":
        # Blue (low walkability) -> Yellow -> Green (high walkability)
        colormap = cm.LinearColormap(colors=["blue", "yellow", "green"],
                                     index=[min_val, mid_val, max_val], vmin=min_val, vmax=max_val)
        colormap.caption = f"Color based on {feature} (Higher is Better)"
    elif feature == "price":
        # Sequential: Yellow (low price) -> Orange -> Red (high price)
        colormap = cm.LinearColormap(colors=["yellow", "orange", "red"],
                                     index=[min_val, mid_val, max_val], vmin=min_val, vmax=max_val)
        colormap.caption = f"Color based on {feature} (Higher Price = Redder)"
    elif feature == "aqi":
        # Green (low AQI - good) -> Yellow -> Red (high AQI - poor)
        colormap = cm.LinearColormap(colors=["green", "yellow", "red"],
                                     index=[min_val, mid_val, max_val], vmin=min_val, vmax=max_val)
        colormap.caption = f"Color based on {feature} (Lower AQI is Better)"
    else: # Default colormap if feature name changes
        colormap = cm.linear.YlGnBu_09 # Example default
        colormap.caption = f"Color based on {feature}"

    # Add points to the map
    for _, row in working_df.iterrows():
        feat_val = row[feature]
        popup_text = f"{feature}: {feat_val:.2f}<br>Price: ${row['price']:.0f}"
        
        folium.CircleMarker(
            location=(row["latitude"], row["longitude"]),
            radius=3,
            popup=popup_text,
            color=colormap(feat_val),
            fill=True,
            fill_opacity=0.7,
            fill_color=colormap(feat_val),
        ).add_to(m)
        
    m.add_child(colormap)
    
    # Save the map to an HTML file
    map_filename = f"map_points_{feature}.html"
    m.save(map_filename)
    all_maps[feature] = m

In [ ]:
all_maps["nwi_score"]

In [ ]:
all_maps["price"]

In [ ]:
all_maps["aqi"]

# Feature Engineering

### Log Transformation

In [ ]:
# Do Log transformation to reduce skewness
working_df["price_log"] = np.log1p(working_df["price"])
working_df["nearest_bus_stop_miles_log"] = np.log1p(working_df["nearest_bus_stop_miles"])
working_df["nearby_bus_stops_log"] = np.log1p(working_df["nearby_bus_stops"])
working_df["nearest_park_miles_log"] = np.log1p(working_df["nearest_park_miles"])


### Standardization

In [ ]:
working_df.columns

In [ ]:
feature_cols = [
    "price_log",
    "aqi",
    "nwi_score",
    "nearest_bus_stop_miles_log",
    "nearby_bus_stops_log",
    "nearby_parks",
    "nearest_park_miles_log"
]
X = working_df[feature_cols].copy()
X.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(X_scaled, columns=feature_cols, index=working_df.index)
df_scaled.describe().round(2)

### Adjust Directionality of Features
* Summary:
  - `price_log`: Has ambiguity in directionality (higher price does not always mean lower QoL, and vice versa), so do not adjust.
  - `aqi`: Higher is worse (higher AQI = lower QoL), so do need adjustment.
  - `nwi_score`: Higher is better (higher score = higher QoL), so do not adjust.
  - `nearest_bus_stop_miles_log`: Higher is worse (further distance = lower QoL), so do need adjustment.
  - `nearby_bus_stops_log`: Higher is better (more stops = higher QoL)
  - `nearby_parks`: Higher is better (more parks = higher QoL)
  - `nearest_park_miles_log`: Higher is worse (further distance = lower QoL), so do need adjustment.
* Adjustments:
    - `aqi`: Invert the sign
    - `nearest_bus_stop_miles_log`: Invert the sign
    - `nearest_park_miles_log`: Invert the sign

In [ ]:
df_scaled["aqi"] = -df_scaled["aqi"]
df_scaled["nearest_bus_stop_miles_log"] = -df_scaled["nearest_bus_stop_miles_log"]
df_scaled["nearest_park_miles_log"] = -df_scaled["nearest_park_miles_log"]

In [ ]:
df_scaled.head()

# PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=len(feature_cols))
pca.fit(df_scaled)

In [ ]:
# Check variance
for i, ratio in enumerate(pca.explained_variance_ratio_, 1):
    print(f"PC{i}: {ratio:.4f}")

In [ ]:
# Examine the component loadings (eigenvectors)
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=[f"PC{i+1}" for i in range(len(feature_cols))]
)
loadings.round(4)

### PCA Analysis
* We applied Principal Component Analysis (PCA) to reduce our 7 standardized Quality of Life indicators into a smaller set of uncorrelated dimensions, called principal components, that capture the most significant patterns of variation in the data
* The analysis revealed 7 principal components. The proportion of total variance explained by each component is [0.50472342 0.17352659 0.11734884 0.09894895 0.05892451 0.03579297
 0.01073472]. 
 * The first component contains the most information about the data, which means its loadings form the best data-driven weights for a one-dimensional Quality of Life index.

# Calculate QoL Index
1. Extract PC1 loadings
2. Take absolute values of loadings
3. Normalize loadings to sum to 1

In [ ]:
pc1 = pca.components_[0]
print(pc1)

In [ ]:
abs_loadings = np.abs(pc1)

In [ ]:
weights = abs_loadings / abs_loadings.sum()
print(weights)

In [ ]:
df_qol = pd.DataFrame(
    {
        "listing_db_id": working_df["listing_db_id"],
        "latitude": working_df["latitude"],
        "longitude": working_df["longitude"],
        "QoL_Index": df_scaled.dot(weights)
    }
)

df_qol.head()

In [ ]:
# Step 6: Visualize the QoL Index Distribution
plt.figure(figsize=(8, 6))
sns.histplot(df_qol["QoL_Index"], kde=True)
plt.title("Distribution of QoL Index")
plt.xlabel("QoL_index")
plt.show()

plt.savefig("final_dist.png", dpi=300, bbox_inches="tight")

In [ ]:
# Normalize QoL Index to range [1, 100]
min_qol = df_qol["QoL_Index"].min()
max_qol = df_qol["QoL_Index"].max()

df_qol["QoL_0_1"] = (df_qol["QoL_Index"] - min_qol) / (max_qol - min_qol)

In [ ]:
# import geopandas as gpd
#
# gdf = gpd.GeoDataFrame(df_qol, geometry=gpd.points_from_xy(df_qol.longitude, df_qol.latitude))
#
# fig, ax = plt.subplots(figsize=(10, 8))
# gdf.plot(
#     column="QoL_0_1",
#     cmap="viridis",
#     legend=True,
#     ax=ax, markersize=10,
#     legend_kwds={
#         "label": "QoL Index (0–1)",
#         "orientation": "vertical",
#         "shrink": 0.6,
#         "pad": 0.02,
#         "aspect": 20
#     }
#     )
# ax.set_title("QoL Index Distribution on Map")
# ax.set_xlabel("Longitude")
# ax.set_ylabel("Latitude")
# plt.show()
#
# plt.show()
# plt.savefig("final.png", dpi=300, bbox_inches="tight")

### Summary of QoL Index Analysis:
- Geospatial patterns (from heatmap): High QoL scores cluster in downtown D.C. due to better transit and parks.

In [ ]:
# Save the final DataFrame to a CSV file
output_filename = "final_rental_listings_with_qol.csv"
df_qol.to_csv(output_filename, index=False)